# Mars Forecast Analyst Case - January Forecast Recommendation

This notebook turns the exploration into a forecast position for the Sales and Supply Chain meeting.

In [ ]:
"""Load reusable forecast functions and generated outputs."""

from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from mars_forecast_case.data import load_supporting_data
from mars_forecast_case.forecast import (
    base_weekly_forecast,
    build_january_scenarios,
    recent_clean_base,
    scenario_frame,
    scenario_summary_frame,
    summarize_history,
)
from mars_forecast_case.pipeline import build_outputs

sns.set_theme(style="whitegrid")
build_outputs(PROJECT_ROOT / "outputs")
weekly, metadata = load_supporting_data()


## Forecast Principle

The operational forecast for the factory should include committed demand, not Sales optimism.
A promotion belongs in the official number only once the customer, mechanics, dates, expected phasing,
and supply feasibility are confirmed.

In [ ]:
def compare_base_reads(df: pd.DataFrame) -> pd.DataFrame:
    """Compare plausible base-demand anchors for January 2022."""

    recent = recent_clean_base(df)
    return pd.DataFrame(
        {
            "base_read": [
                "P01 2021 estimated ordered demand",
                "P13 2021 estimated ordered demand",
                "Recent clean weekly median x 4",
                "Recent clean weekly mean x 4",
            ],
            "forecast_k_cases": [
                df.loc[df["period"].eq("P01"), "estimated_ordered_k_cases"].sum(),
                df.loc[df["period"].eq("P13"), "estimated_ordered_k_cases"].sum(),
                recent["estimated_ordered_k_cases"].median() * 4,
                recent["estimated_ordered_k_cases"].mean() * 4,
            ],
            "comment": [
                "Direct January analogy but one year old.",
                "Latest clean four-week period with normal service.",
                "Robust recent base, ignores weekly phasing.",
                "Recent base but more sensitive to high weeks.",
            ],
        }
    )


base_reads = compare_base_reads(weekly)
base_reads


In [ ]:
"""Build weekly January scenarios from the reusable model."""

scenarios = build_january_scenarios(weekly)
scenario_weekly = scenario_frame(scenarios)
scenario_summary = scenario_summary_frame(scenarios)
display(scenario_weekly)
display(scenario_summary)


In [ ]:
"""Visualize the weekly shape of each January scenario."""

fig, ax = plt.subplots(figsize=(10, 5))
for scenario, data in scenario_weekly.groupby("scenario"):
    ax.plot(data["week_of_period"], data["forecast_k_cases"], marker="o", linewidth=2.4, label=scenario)
ax.set_title("January 2022 weekly forecast scenarios")
ax.set_xlabel("Week of period")
ax.set_ylabel("'000 cases")
ax.legend(frameon=False)
plt.show()


In [ ]:
"""Visualize scenario totals for the Sales and Supply Chain sign-off discussion."""

colors = ["#007A78", "#9E1B32", "#F2A900"]
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.barh(scenario_summary["scenario"], scenario_summary["total_forecast_k_cases"], color=colors)
for idx, row in scenario_summary.iterrows():
    ax.text(row["total_forecast_k_cases"] + 20, idx, f"{row['total_forecast_k_cases']:.0f}", va="center")
ax.set_title("Recommended forecast separates base from promo upside")
ax.set_xlabel("'000 cases")
plt.show()


In [ ]:
"""Create the final meeting recommendation and information request."""

metrics = summarize_history(weekly)
signed_total = float(
    scenario_summary.loc[
        scenario_summary["scenario"].eq("Signed two-week promo"),
        "total_forecast_k_cases",
    ].iloc[0]
)

recommendation = pd.DataFrame(
    {
        "topic": [
            "Factory forecast to sign off",
            "Promotion gate",
            "Additional information needed",
            "Sales follow-up",
        ],
        "position": [
            f"Use {metrics['p13_estimated_ordered_k_cases']:.0f}k cases as the no-promo January baseline.",
            f"Move to {signed_total:.0f}k cases only for a signed two-week event with confirmed weekly phasing.",
            "Customer, mechanic, discount/display support, committed orders, cannibalization, and supply feasibility.",
            "Ask Sales to own the customer volume assumption and review actual orders/casefill weekly during January.",
        ],
    }
)
recommendation


## Forecast Decision

Recommend **856k cases** as the January 2022 factory baseline.
If Sales secures a confirmed two-week Customer 1-like promotion, prepare the factory for **1,144k cases**,
with W2-W3 as the constrained weeks. Keep the larger Sales upside sensitivity as a capacity-risk discussion,
not as the default operational forecast.